## 5.1 损失函数 - 二分类交叉熵 （Binary Cross Entropy Loss）
在神经网络训练过程中，我们需要一个函数来衡量：
`模型预测结果与真实标签之间的差距`

这个函数就叫做：

`损失函数（Loss Function）`

不同任务会使用不同的损失函数，例如：
* 回归任务 → MSELoss
* 二分类任务 → Binary Cross Entropy
* 多分类任务 → Cross Entropy

本节我们重点学习：

`二分类交叉熵损失（Binary Cross Entropy Loss）`

#### 1. 二分类任务的输出是什么？
在很多机器学习任务中，我们需要预测的结果只有两种可能：
* 0
* 1

例如：
* 垃圾邮件识别（Spam / Not Spam）
* 疾病预测（Sick / Healthy）
* 欺诈检测（Fraud / Normal）

这种问题叫做：

`二分类问题（Binary Classification）`

在二分类任务中，最常使用的损失函数是：

`Binary Cross Entropy（BCE）`

在二分类问题中，模型通常只输出：一个数值

例如：

`z = 2.3`

这个值叫：

`logit`

⚠️注意：
* 这个值 不是概率
* 需要配合激活函数 Sigmoid 转换之后得到概率


##### 1.1 为什么需要 Sigmoid？
为了把 logit 转换为概率，我们需要使用：

`Sigmoid 函数`

Sigmoid 会把输出转换为：`(0,1)`之间的数值。

例如：
```
z = 2.3
σ(z) = 0.91
```
这可以理解为：

`P(y=1 | x) = 0.91`

样本属于正类的概率。

#### 2. Binary Cross Entropy 公式
Binary Cross Entropy 的公式是：

`Loss = -(y * log(p) + (1-y) * log(1-p))`

其中：
* y → 真实标签（0 或 1）
* p → 预测概率

##### 2.1 当真实标签为 1
如果：

`y = 1`

公式变为：

`Loss = -log(p)`

例如：
```
p	loss
0.9	0.105
0.5	0.693
0.1	2.302
```
正确类别概率越小，loss 越大。


##### 2.2 当真实标签为 0
如果：

`y = 0`

公式变为：

`Loss = -log(1-p)`

例如：
```
p	loss
0.1	0.105
0.5	0.693
0.9	2.302
```

如果真实是 0，但模型预测接近 1，loss 会很大。


#### 3. 二分类交叉熵的核心思想
Binary Cross Entropy 的核心思想是：

`如果模型预测的概率越接近真实标签，损失就越小。`

例如：

真实标签：

`y = 1`

✅ 情况1：预测很好

`p = 0.92`

loss 很小。


🙅 情况2：预测很差

`p = 0.10`

loss 很大。

#### 4. BCE 的直观理解
Binary Cross Entropy 本质上是在衡量：

`预测概率 vs 真实标签`

如果：
* 预测概率接近真实标签

loss 就会：
* 接近 0

如果：
* 预测概率与真实标签相差很大

loss 就会：
* 很大

#### 5. PyTorch 中的 BCE
在 PyTorch 中有两个相关的损失函数：

##### 5.1 BCELoss
`nn.BCELoss()`

要求：模型输出必须是概率

因此模型结构必须是：

`Linear → Sigmoid → BCELoss`

##### 5.2 BCEWithLogitsLoss（✅ 推荐）
`nn.BCEWithLogitsLoss()`

这个函数内部已经包含：

`Sigmoid + BCE`

因此：**⚠️ 不要自己再加 Sigmoid。**

✅ 正确结构是：

`Linear → BCEWithLogitsLoss`

🙅错误写法

`Linear + Sigmoid + BCEWithLogitsLoss`

##### 6, Pytorch 实现

##### 6.1 构建模型

In [2]:
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(10, 5)
        self.Relu = nn.ReLU()
        self.layer_2 = nn.Linear(5, 1)
        self.Sigmoid = nn.Sigmoid() # 注意此时输出层已经指定了激活函数为Sigmoid，所以损失函数应该使用BCELoss，而不是BCEWithLogitsLoss

    def forward(self, x):
        x = self.layer_1(x)
        x = self.Relu(x)
        x = self.layer_2(x)
        x = self.Sigmoid(x)
        return x

##### 6.2 准备数据

In [3]:
X = torch.randn(3, 10)  # 输入数据，3个样本，每个样本10个特征
y = torch.tensor([[1.0], [0.0], [1.0]])  # 目标标签，3个样本的二分类标签

##### 6.3 使用 Sigmoid + BCELoss 训练模型

In [4]:
model = MLP()
# 定义损失函数
criterion = nn.BCELoss()  # 二分类交叉熵损失函数，
# 定义优化器
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(100):
    y_pred = model(X)  # 前向传播
    loss = criterion(y_pred, y)  # 计算损失
    loss.backward()  # 反向传播
    optimizer.step()  # 更新参数
    optimizer.zero_grad()  # 清零梯度
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')


Epoch [10/100], Loss: 0.6460
Epoch [20/100], Loss: 0.6454
Epoch [30/100], Loss: 0.6448
Epoch [40/100], Loss: 0.6443
Epoch [50/100], Loss: 0.6438
Epoch [60/100], Loss: 0.6433
Epoch [70/100], Loss: 0.6429
Epoch [80/100], Loss: 0.6425
Epoch [90/100], Loss: 0.6421
Epoch [100/100], Loss: 0.6418
